# QLoRA Fine-Tuning with Unsloth — Qwen2.5-3B-Instruct
**Model:** `unsloth/Qwen2.5-3B-Instruct-bnb-4bit`  
**Dataset:** `philschmid/gretel-synthetic-text-to-sql`  
**Task:** Text-to-SQL  
**Optimized for:** Google Colab T4 GPU (16 GB VRAM)


## 1. Install Dependencies

In [1]:
# Unsloth handles QLoRA, LoRA, and fast inference in one package
%pip install unsloth
%pip install --upgrade trl datasets accelerate peft bitsandbytes tensorboard

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

## 2. Hugging Face Login

In [2]:
from google.colab import userdata
from huggingface_hub import login

# Store your HF token in Colab Secrets (key = HF_TOKEN)
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

## 3. GPU Check

In [3]:
import torch

name = torch.cuda.get_device_name(0)
cc_major, cc_minor = torch.cuda.get_device_capability(0)
supports_bf16 = cc_major >= 8

print(f"GPU            : {name}")
print(f"Compute Cap    : {cc_major}.{cc_minor}")
print(f"bfloat16       : {'✅ Yes' if supports_bf16 else '❌ No — will use float16'}")
print(f"Total VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# T4 = float16; A100/L4 = bfloat16
torch_dtype = torch.bfloat16 if supports_bf16 else torch.float16
print(f"Using dtype    : {torch_dtype}")

GPU            : Tesla T4
Compute Cap    : 7.5
bfloat16       : ❌ No — will use float16
Total VRAM     : 14.6 GB
Using dtype    : torch.float16


## 4. Load & Prepare Dataset

In [4]:
from datasets import load_dataset

# Load same dataset as original notebook
dataset = load_dataset("philschmid/gretel-synthetic-text-to-sql", split="train")
dataset = dataset.shuffle(seed=42).select(range(12500))

print(f"Dataset size: {len(dataset)}")
print(f"Columns     : {dataset.column_names}")

README.md:   0%|          | 0.00/737 [00:00<?, ?B/s]

synthetic_text_to_sql_train.snappy.parqu(…):   0%|          | 0.00/32.4M [00:00<?, ?B/s]

synthetic_text_to_sql_test.snappy.parque(…):   0%|          | 0.00/1.90M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5851 [00:00<?, ? examples/s]

Dataset size: 12500
Columns     : ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation']


In [5]:
# System message — same as original
system_message = """You are a text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA."""

# User prompt — same as original
user_prompt = """Given the <USER_QUERY> and the <SCHEMA>, generate the corresponding SQL command to retrieve the desired data, considering the query's syntax, semantics, and schema constraints.

<SCHEMA>
{context}
</SCHEMA>

<USER_QUERY>
{question}
</USER_QUERY>
"""

def create_conversation(sample):
    return {
        "messages": [
            {"role": "system", "content": system_message},
            {"role": "user",   "content": user_prompt.format(
                question=sample["sql_prompt"],
                context=sample["sql_context"]
            )},
            {"role": "assistant", "content": sample["sql"]}
        ]
    }

# Convert to conversation format
dataset = dataset.map(create_conversation, remove_columns=dataset.features, batched=False)

# Split: 10,000 train / 2,500 test
dataset = dataset.train_test_split(test_size=2500 / 12500)

print(f"Train samples : {len(dataset['train'])}")
print(f"Test  samples : {len(dataset['test'])}")
print("\nSample entry:")
print(dataset["train"][0])

Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

Train samples : 10000
Test  samples : 2500

Sample entry:
{'messages': [{'role': 'system', 'content': 'You are a text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.'}, {'role': 'user', 'content': "Given the <USER_QUERY> and the <SCHEMA>, generate the corresponding SQL command to retrieve the desired data, considering the query's syntax, semantics, and schema constraints.\n\n<SCHEMA>\nCREATE TABLE Events (event_name VARCHAR(255), attendee_age INT); INSERT INTO Events (event_name, attendee_age) VALUES ('Art Exhibition', 38), ('Dance Performance', 32), ('Music Concert', 41);\n</SCHEMA>\n\n<USER_QUERY>\nFind the event with the highest average attendee age.\n</USER_QUERY>\n"}, {'role': 'assistant', 'content': 'SELECT event_name, AVG(attendee_age) FROM Events GROUP BY event_name ORDER BY AVG(attendee_age) DESC LIMIT 1;'}]}


## 5. Load Model with Unsloth (replaces BitsAndBytes manual setup)

In [6]:
from unsloth import FastLanguageModel
import torch

# ── Config ────────────────────────────────────────────────────────────────────
model_id       = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"  # pre-quantized, no gating
max_seq_length = 2048   # Unsloth handles long sequences efficiently on T4
load_in_4bit   = True   # QLoRA 4-bit — keeps VRAM ~6-7 GB for this model

# ── Load ──────────────────────────────────────────────────────────────────────
# Unsloth automatically:
#   • applies NF4 quantization (same as BitsAndBytesConfig in original)
#   • patches attention for 2x faster training
#   • selects float16 for T4, bfloat16 for A100/L4
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = model_id,
    max_seq_length = max_seq_length,
    dtype          = None,         # auto: float16 on T4, bfloat16 on Ampere+
    load_in_4bit   = load_in_4bit,
)

print("✅ Model loaded")
print(f"Free VRAM after load: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Model loaded
Free VRAM after load: 12.43 GB


## 6. Attach QLoRA Adapters with Unsloth

In [7]:
# Unsloth's get_peft_model replaces LoraConfig + get_peft_model from PEFT
# It applies optimized LoRA kernels automatically
model = FastLanguageModel.get_peft_model(
    model,
    r                  = 16,    # LoRA rank — same as original
    lora_alpha         = 16,    # same as original
    lora_dropout       = 0,     # Unsloth recommendation: 0 is optimal
    bias               = "none",
    target_modules     = [      # same coverage as "all-linear" in original
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing = "unsloth",  # saves ~30% VRAM vs standard
    random_state       = 42,
    use_rslora         = False,
    loftq_config       = None,
)

print("QLoRA adapters attached")
model.print_trainable_parameters()
# Expected: ~1-2% trainable params — correct for QLoRA

Unsloth 2026.4.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


QLoRA adapters attached
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## 7. Training Config

In [8]:
from trl import SFTConfig

# Detect dtype for fp16/bf16 flags
supports_bf16 = torch.cuda.get_device_capability()[0] >= 8

args = SFTConfig(
    output_dir                 = "qwen-text-to-sql",   # changed from gemma-text-to-sql
    packing                    = True,                 # same as original
    num_train_epochs           = 3,                    # same as original
    max_seq_length             = 2048,                 # ✅ increased from 512 — Unsloth handles it
    per_device_train_batch_size= 2,                    # ✅ increased from 1 — more VRAM headroom
    gradient_accumulation_steps= 4,                    # ✅ effective batch = 8
    gradient_checkpointing     = True,                 # same as original
    optim                      = "adamw_8bit",         # ✅ saves ~1GB vs adamw_torch_fused
    logging_steps              = 10,                   # same as original
    save_strategy              = "epoch",              # same as original
    learning_rate              = 2e-4,                 # same as original
    fp16                       = not supports_bf16,    # float16 on T4
    bf16                       = supports_bf16,        # bfloat16 on A100/L4
    max_grad_norm              = 0.3,                  # same as original
    warmup_steps               = 10,                   # ✅ replaces deprecated warmup_ratio
    lr_scheduler_type          = "constant",           # same as original
    push_to_hub                = False,                # ✅ disabled — save locally
    report_to                  = "tensorboard",        # same as original
    dataset_kwargs             = {
        "add_special_tokens": False,                   # same as original
        "append_concat_token": True,                   # same as original
    },
)

print("✅ SFTConfig ready")

✅ SFTConfig ready


## 8. Create Trainer

In [12]:
from trl import SFTTrainer

def formatting_func(example):
    # apply_chat_template can return list or str depending on input shape
    # force it to always be a plain string first
    messages = example["messages"]

    # If messages is a list of lists (batched), handle first item
    if isinstance(messages[0], list):
        messages = messages[0]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,              # return string, not token ids
        add_generation_prompt=False,
    )

    # Ensure it's a plain string before wrapping
    if isinstance(text, list):
        text = text[0]

    return [str(text)]   # Unsloth needs a list of strings

trainer = SFTTrainer(
    model            = model,
    args             = args,
    train_dataset    = dataset["train"],
    formatting_func  = formatting_func,
    processing_class = tokenizer,
)

print("✅ Trainer ready")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10000 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/12 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
✅ Trainer ready


## 9. Clear Cache & Train

In [13]:
import gc, os

# Prevent CUDA memory fragmentation (from your error message)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"Free VRAM before training : {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")
print(f"Used VRAM before training : {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Free VRAM before training : 12.30 GB
Used VRAM before training : 2.07 GB


In [14]:
# Train
trainer_stats = trainer.train()

# Summary
print(f"\n✅ Training complete")
print(f"Total steps    : {trainer_stats.global_step}")
print(f"Training loss  : {trainer_stats.training_loss:.4f}")
print(f"Peak VRAM used : {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6 | Num Epochs = 3 | Total steps = 3
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss


Unsloth: Restored added_tokens_decoder metadata in qwen-text-to-sql/checkpoint-1/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen-text-to-sql/checkpoint-2/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen-text-to-sql/checkpoint-3/tokenizer_config.json.



✅ Training complete
Total steps    : 3
Training loss  : 2.0738
Peak VRAM used : 2.95 GB


## 10. Save Model

In [15]:
# Save LoRA adapters locally (~100MB — much smaller than full model)
trainer.save_model("qwen-text-to-sql")
tokenizer.save_pretrained("qwen-text-to-sql")
print("✅ LoRA adapters saved to ./qwen-text-to-sql")

Unsloth: Restored added_tokens_decoder metadata in qwen-text-to-sql/tokenizer_config.json.


✅ LoRA adapters saved to ./qwen-text-to-sql


In [16]:
# Backup to Google Drive (prevents loss on Colab session reset)
from google.colab import drive
import shutil

drive.mount('/content/drive')

dest = "/content/drive/MyDrive/qwen-text-to-sql"
shutil.copytree("qwen-text-to-sql", dest, dirs_exist_ok=True)
print(f"✅ Model backed up to Google Drive: {dest}")

Mounted at /content/drive
✅ Model backed up to Google Drive: /content/drive/MyDrive/qwen-text-to-sql


## 11. Free Memory & Merge Model

In [17]:
# Free memory before merging
del trainer
gc.collect()
torch.cuda.empty_cache()
print(f"Free VRAM after cleanup: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")

Free VRAM after cleanup: 12.22 GB


In [18]:
from unsloth import FastLanguageModel

# Reload and merge LoRA into base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "qwen-text-to-sql",   # your saved adapter
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)

# Merge LoRA weights into base model and save full merged model
model.save_pretrained_merged(
    "qwen-text-to-sql-merged",
    tokenizer,
    save_method = "merged_16bit",   # full precision merged model
)
print("✅ Merged model saved to ./qwen-text-to-sql-merged")

# Optional: save as GGUF for local inference with Ollama / LM Studio
# model.save_pretrained_gguf("qwen-gguf", tokenizer, quantization_method="q4_k_m")
# print("✅ GGUF model saved")

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in qwen-text-to-sql-merged/tokenizer_config.json.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [01:56<00:00, 58.43s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:10<00:00, 95.19s/it]


Unsloth: Merge process complete. Saved to `/content/qwen-text-to-sql-merged`
✅ Merged model saved to ./qwen-text-to-sql-merged


## 12. Test Inference

In [19]:
import torch, re
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from random import randint

# Load fine-tuned model for inference
infer_model = AutoModelForCausalLM.from_pretrained(
    "qwen-text-to-sql",
    device_map  = "auto",
    torch_dtype = torch.float16,
    attn_implementation = "eager",
)
infer_tokenizer = AutoTokenizer.from_pretrained("qwen-text-to-sql")

pipe = pipeline("text-generation", model=infer_model, tokenizer=infer_tokenizer)
print("✅ Inference pipeline ready")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

✅ Inference pipeline ready


In [21]:
from unsloth import FastLanguageModel
from transformers import pipeline
from random import randint
import re

# ✅ Switch model to Unsloth inference mode BEFORE creating pipeline
FastLanguageModel.for_inference(model)

# Create pipeline using the SAME model and tokenizer (no reload needed)
pipe = pipeline(
    "text-generation",
    model     = model,
    tokenizer = tokenizer,
)

print("✅ Inference pipeline ready")

✅ Inference pipeline ready


In [22]:
# Pick a random test sample
rand_idx    = randint(0, len(dataset["test"]) - 1)
test_sample = dataset["test"][rand_idx]

# Stop tokens for Qwen
stop_token_ids = [tokenizer.eos_token_id]

# Build prompt — system + user only, no assistant turn
prompt = tokenizer.apply_chat_template(
    test_sample["messages"][:2],
    tokenize              = False,
    add_generation_prompt = True,
)

# Generate — clean config, no deprecated mixing
from transformers import GenerationConfig

outputs = pipe(
    prompt,
    max_new_tokens = 512,
    do_sample      = False,
    eos_token_id   = stop_token_ids,
    pad_token_id   = tokenizer.eos_token_id,  # silence padding warning
)

# Parse and display
schema = re.search(r'<SCHEMA>\n(.*?)\n</SCHEMA>',
                   test_sample["messages"][1]["content"], re.DOTALL)
query  = re.search(r'<USER_QUERY>\n(.*?)\n</USER_QUERY>',
                   test_sample["messages"][1]["content"], re.DOTALL)

print("=" * 60)
print("SCHEMA:")
print(schema.group(1).strip() if schema else "N/A")
print("\nUSER QUERY:")
print(query.group(1).strip() if query else "N/A")
print("\nEXPECTED SQL:")
print(test_sample["messages"][2]["content"])
print("\nGENERATED SQL:")
print(outputs[0]["generated_text"][len(prompt):].strip())
print("=" * 60)

Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'pad_token_id', 'do_sample', 'max_new_tokens', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SCHEMA:
CREATE TABLE community_health_workers (region VARCHAR(255), workers INT); INSERT INTO community_health_workers (region, workers) VALUES ('Northeast', 200), ('Southeast', 250), ('Midwest', 180), ('West', 300);

USER QUERY:
What is the minimum number of community health workers by region?

EXPECTED SQL:
SELECT region, MIN(workers) FROM community_health_workers GROUP BY region;

GENERATED SQL:
SELECT MIN(workers) FROM community_health_workers;


In [23]:
from unsloth import FastLanguageModel
from transformers import pipeline
import re

# Make sure model is in inference mode
FastLanguageModel.for_inference(model)

pipe = pipeline(
    "text-generation",
    model     = model,
    tokenizer = tokenizer,
)

# ── System message ────────────────────────────────────────────────
system_message = """You are a text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA."""

user_prompt_template = """Given the <USER_QUERY> and the <SCHEMA>, generate the corresponding SQL command to retrieve the desired data, considering the query's syntax, semantics, and schema constraints.

<SCHEMA>
{schema}
</SCHEMA>

<USER_QUERY>
{question}
</USER_QUERY>
"""

# ── Chat function ─────────────────────────────────────────────────
def generate_sql(schema: str, question: str) -> str:
    messages = [
        {"role": "system",    "content": system_message},
        {"role": "user",      "content": user_prompt_template.format(
            schema=schema,
            question=question
        )},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize              = False,
        add_generation_prompt = True,
    )

    outputs = pipe(
        prompt,
        max_new_tokens = 256,
        do_sample      = False,
        eos_token_id   = [tokenizer.eos_token_id],
        pad_token_id   = tokenizer.eos_token_id,
    )

    generated = outputs[0]["generated_text"][len(prompt):].strip()
    return generated

# ── Interactive loop ──────────────────────────────────────────────
print("=" * 60)
print("   TEXT-TO-SQL CHATBOT — Qwen2.5-3B Fine-Tuned")
print("=" * 60)\
print("Type 'quit' to exit | Type 'example' for a sample schema\n")

EXAMPLE_SCHEMA = "CREATE TABLE employees (id INT, name VARCHAR(100), department VARCHAR(50), salary INT, hire_date DATE)"

while True:
    print("\n" + "-" * 60)

    # Get schema
    schema_input = input("📋 Enter your schema (or 'example' to use sample): ").strip()

    if schema_input.lower() == "quit":
        print(" Exiting chatbot.")
        break

    if schema_input.lower() == "example":
        schema_input = EXAMPLE_SCHEMA
        print(f"   Using: {schema_input}")

    if not schema_input:
        print("Schema cannot be empty.")
        continue

    # Get question
    question_input = input(" Enter your question: ").strip()

    if question_input.lower() == "quit":
        print(" Exiting chatbot.")
        break

    if not question_input:
        print("  Question cannot be empty.")
        continue

    # Generate
    print("\nGenerating SQL...")
    try:
        sql = generate_sql(schema_input, question_input)
        print("\n Generated SQL:")
        print("   " + sql)
    except Exception as e:
        print(f"Error: {e}")

   TEXT-TO-SQL CHATBOT — Qwen2.5-3B Fine-Tuned
Type 'quit' to exit | Type 'example' for a sample schema


------------------------------------------------------------
📋 Enter your schema (or 'example' to use sample): Show all employees with salary above 50000
 Enter your question: exit


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generating SQL...

 Generated SQL:
   It seems like there might be a misunderstanding or an error in the provided information. The schema provided is "Show all employees with salary above 50000", but the user query is "exit". Since no actual query was given, I'll assume you want to retrieve all employees with a salary above 50000 using a typical SQL query. Here’s how you can write that query:

```sql
SELECT * FROM Employees WHERE Salary > 50000;
```

This assumes that there is a table named `Employees` with a column named `Salary`. If your schema or table names differ, please provide the correct details.

------------------------------------------------------------
📋 Enter your schema (or 'example' to use sample): CREATE TABLE users (id INT, name VARCHAR, age INT, email VARCHAR)
 Enter your question: Show all users over 30


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generating SQL...

 Generated SQL:
   SELECT * FROM users WHERE age > 30;

------------------------------------------------------------
📋 Enter your schema (or 'example' to use sample): CREATE TABLE sales (id INT, product VARCHAR, amount FLOAT, region VARCHAR)
 Enter your question: What is the total amount per region?


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generating SQL...

 Generated SQL:
   SELECT region, SUM(amount) FROM sales GROUP BY region;

------------------------------------------------------------
📋 Enter your schema (or 'example' to use sample): CREATE TABLE orders (id INT, customer VARCHAR, status VARCHAR, 
 Enter your question: How many orders are pending?


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generating SQL...

 Generated SQL:
   SELECT COUNT(*) FROM orders WHERE status = 'pending';

------------------------------------------------------------
📋 Enter your schema (or 'example' to use sample): exit
 Enter your question: quit
 Exiting chatbot.
